In [116]:
#1. data loading, encoding and decoding lambdas
with open('./demo-input.txt', 'r', encoding='utf-8') as f: 
    text = f.read()

# unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)

display({'text_length': len(text), 'chars': ''.join(chars), 'vocab_size': vocab_size})

# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) } # string to index
itos = { i:ch for i,ch in enumerate(chars) } # index to string
# character-level encoder and decoder
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# encode-decode test
print(encode("nel mezzo del camin"),"<=>", decode(encode("nel mezzo del camin")))

{'text_length': 1185518,
 'chars': "\n !'(),-.0123456789:;>?ABCDEFGHIJKLMNOPQRSTUVXZ[\\]abcdefghijlmnopqrstuvwxyz§«»ÈËÏÖàâäèéêëìïòóöùü–—‘’“”•…′",
 'vocab_size': 105}

[62, 54, 60, 1, 61, 54, 74, 74, 63, 1, 53, 54, 60, 1, 52, 50, 61, 58, 62] <=> nel mezzo del camin


In [129]:
#2. model definition
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
batch_size = 64 # how many independent sequences will we process in parallel?
block_size = 256 # what is the maximum context length for predictions?
max_iters = 5_000 # maximum number of training iterations
eval_iters = 100 # how many iterations to evalute loss on
learning_rate = 3e-4 # learning rate for optimizer
device = 'cuda' if torch.cuda.is_available() else 'cpu'
n_embd = 384 # the dimensionality of the embeddings
n_head = 6 # number of attention heads
n_layer = 6 # number of transformer blocks
dropout = 0.2 # dropout rate
# ------------

torch.manual_seed(42)

class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # input of size (batch, time-step, channels)
        # output of size (batch, time-step, head size)
        B,T,C = x.shape
        k = self.key(x)   # (B,T,hs)
        q = self.query(x) # (B,T,hs)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5 # (B, T, hs) @ (B, hs, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,hs)
        out = wei @ v # (B, T, T) @ (B, T, hs) -> (B, T, hs)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class TinyLM(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

        # init
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

In [130]:
model = TinyLM()
params = sum(p.numel() for p in model.parameters())
tokens = len(encode(text))
print(params/1e6, 'M parameters')
print('tokens:', tokens)
chinchilla =  params / tokens
print(f"Chinchilla ratio: {chinchilla:.2f} (ideal: 20)")

10.819689 M parameters
tokens: 1185518
Chinchilla ratio: 9.13 (ideal: 20)


In [120]:
#3a. train: skip & load model (step 4)
import torch

model = TinyLM()
model = model.to(device)

# data loading
with open('./demo-input.txt', 'r', encoding='utf-8') as f: 
    text = f.read()
    text = text.replace(('’'), "'")

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad() # disable gradient calculation 
def estimate_loss():
    out = {}
    model.eval() # set model to evaluation mode, turn off dropout layers
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train() # re-enable training mode, turn on dropout layers
    return out

# Train and test splits with shuffled chunks by paragraph/stanza
data = torch.tensor(encode(text), dtype=torch.long)
# Split text by empty lines (paragraphs/stanzas)
paragraphs = text.split('\n\n')  # split on double newline
paragraph_tokens = [torch.tensor(encode(p), dtype=torch.long) for p in paragraphs if p.strip()]
# Shuffle paragraphs
torch.manual_seed(42)
indices = torch.randperm(len(paragraph_tokens))
shuffled_paragraphs = [paragraph_tokens[i] for i in indices]
# 80/20 split on shuffled paragraphs
n = int(0.8 * len(shuffled_paragraphs))
train_paragraphs = shuffled_paragraphs[:n]
val_paragraphs = shuffled_paragraphs[n:]
# Concatenate back
train_data = torch.cat(train_paragraphs)
val_data = torch.cat(val_paragraphs)

# create optimizer with learning rate
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
#scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_iters)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_iters == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    #scheduler.step() # decay learning rate over time

    if loss.item() < 1.5: # early stopping criterion
        print("stopping training.")
        break


step 0: train loss 4.7345, val loss 4.7346
step 100: train loss 2.3821, val loss 2.4009
step 200: train loss 2.3077, val loss 2.3289
step 300: train loss 2.2364, val loss 2.2548
step 400: train loss 2.0192, val loss 2.0375
step 500: train loss 1.8291, val loss 1.8539
step 600: train loss 1.7140, val loss 1.7470
step 700: train loss 1.6203, val loss 1.6641
step 800: train loss 1.5412, val loss 1.5925
step 900: train loss 1.4764, val loss 1.5407
step 1000: train loss 1.4300, val loss 1.5020
step 1100: train loss 1.3956, val loss 1.4746
step 1200: train loss 1.3588, val loss 1.4443
step 1300: train loss 1.3341, val loss 1.4274
step 1400: train loss 1.3081, val loss 1.4084
step 1500: train loss 1.2852, val loss 1.3868
step 1600: train loss 1.2650, val loss 1.3778
step 1700: train loss 1.2454, val loss 1.3680
step 1800: train loss 1.2297, val loss 1.3610
step 1900: train loss 1.2122, val loss 1.3533
step 2000: train loss 1.2029, val loss 1.3472
step 2100: train loss 1.1831, val loss 1.3379


In [125]:
#3b. save the model
torch.save(model.state_dict(), './demo-model.pth')
#zip 
from zipfile import ZipFile
import zipfile
with ZipFile('demo-model.zip', 'w', compression=zipfile.ZIP_DEFLATED) as zipf:
    zipf.write('demo-model.pth')
# remove the unzipped model file to save space
import os
os.remove('demo-model.pth')

In [126]:
#4. load the model
from zipfile import ZipFile
#unzip
with ZipFile('demo-model.zip', 'r') as zipf:
    zipf.extractall()
# load the model
model = TinyLM()
model.load_state_dict(torch.load('./demo-model.pth'))
model = model.to(device)
model.eval() # swith to inference mode, so dropout or batchnorm is disabled (used only during training)
print(sum(p.numel() for p in model.parameters())/1e6, 'M parameters')

10.819689 M parameters


In [127]:
#5. generate
# - with starting sequence
context = torch.tensor(encode("A la bottega sull'intelligenza"), dtype=torch.long, device=device).unsqueeze(0) # unsqueeze to add batch dimension from (T,) to (1, T)
print(decode(model.generate(context, max_new_tokens=500)[0].tolist()))

print("\n=====\n")

# - with no starting context
context = torch.zeros((1, 1), dtype=torch.long, device=device) # starting context
print(decode(model.generate(context, max_new_tokens=500)[0].tolist()))

A la bottega sull'intelligenza e la bellezza del sommo a cu' io parlo facea; che più caderà quella al punto parlare, dove molte volte volte donne, e dicea alquanti parole a li visibili che pur mostraro, ne la presenza ritornare. Onde vedemo queste parole dice, poi che comincia: Nel capitol di cotale imagine s'aggira, cioè lasciare canzone. 3. La gente che risponde si può conchiudere del suo frutto, è risplendendo. Per che la nobile sentenza de l'anima ricchezza del mio amore, ma quelle è puramente di tempo a l'anima sua, non

=====


  quel beato seme s'apre, che l'una e l'altra giace.  Vedi natura; e lascia la luce
  per la bella disposizion quell' anni.  Io era: «Vevita è questa?
  già spiranza ch'ancor mi viien dir per lor padre;
  ch'è mal diservo del tutto russi».  Tant' è l'un de' miravan lagni amorosi,
  dicendo che la mia sentenzion fue
  de l'orto a che tutti li altri musi,  dal proceder di questo sermone: ond' ello sguardo
  non ti parea percossi a l'ultimo tutto
  tutto grid